# Step 2b: Merge Per-Block VCFs

This notebook takes the per-block VCFs referenced in `blocks/results.json`, transforms block-relative coordinates to absolute coordinates, concatenates them, and writes:
- `merged_variants.vcf.gz` + `.tbi`
- `all_blocks.bed`

## Context
- Each block record in `results.json` has a `variants` field pointing to a per-block VCF
- Block VCFs use relative coordinates within the block region
- We need to transform these to absolute coordinates on the parent assembly
- The input VCFs are pre-filtered (step 2 was run with `--ignore-variants-in-regions`)

## Required inputs
- `blocks/results.json` — block records with variant paths
- `variants/` — per-block VCF files
- `genomes/` — per-chromosome assemblies with `.fai` index files

## Required tools
- `bcftools`, `bgzip`, `tabix`, `bedtools`, `awk`, `sort`

## 1. Setup

Import libraries and define helper functions.

In [27]:
import argparse
import gzip
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union

# Try to import pandas for nicer display (optional)
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("Note: pandas not available, using basic display")

# Configuration
VERBOSE = True

# Default paths (HPC project directory)
PROJECT_ROOT = Path('/home/meta/tomsko/mutation_rates/')
RESULTS_JSON = PROJECT_ROOT / 'blocks' / 'results.json'
VARIANTS_DIR = PROJECT_ROOT / 'variants'
GENOMES_DIR = PROJECT_ROOT / 'genomes'
SAMPLE = 'PAN027'
OUT_VCF = PROJECT_ROOT / 'merged_variants.vcf.gz'
OUT_BED = PROJECT_ROOT / 'all_blocks.bed'
THREADS = int(os.environ.get('PBS_NCPUS', 8))

In [28]:
def run(cmd: List[str], check: bool = True, capture: bool = True, **kwargs) -> subprocess.CompletedProcess:
    """Run a shell command with verbose logging and error handling."""
    cmd_str = " ".join(shlex.quote(str(c)) for c in cmd)
    if VERBOSE:
        print(f"[RUN] {cmd_str}", file=sys.stderr)
    
    has_stdout = 'stdout' in kwargs
    has_stderr = 'stderr' in kwargs
    if has_stdout or has_stderr:
        kwargs.setdefault('text', True)
        result = subprocess.run(cmd, check=False, **kwargs)
    elif capture:
        result = subprocess.run(cmd, check=False, capture_output=True, text=True, **kwargs)
    else:
        result = subprocess.run(cmd, check=False, **kwargs)
    
    if result.returncode != 0:
        print(f"[ERROR] Command failed with exit code {result.returncode}", file=sys.stderr)
        if result.stdout:
            print(f"[STDOUT]\n{result.stdout}", file=sys.stderr)
        if result.stderr:
            print(f"[STDERR]\n{result.stderr}", file=sys.stderr)
        if check:
            raise subprocess.CalledProcessError(result.returncode, cmd_str)
    return result


def q(s: str) -> str:
    """Shell-quote a string for safe command construction."""
    return shlex.quote(str(s))


def validate_identifier(name: str, identifier_type: str) -> None:
    """Validate sample/chromosome identifiers with whitelist regex."""
    if not re.match(r'^[A-Za-z0-9_.-]+$', name):
        raise ValueError(f"Invalid {identifier_type} identifier '{name}': must match ^[A-Za-z0-9_.-]+$")


def check_required_tools() -> None:
    """Check that all required external tools are available."""
    required = ['bcftools', 'bgzip', 'tabix', 'bedtools', 'awk', 'sort']
    missing = []
    for tool in required:
        result = subprocess.run(['which', tool], capture_output=True, text=True)
        if result.returncode != 0:
            missing.append(tool)
    if missing:
        raise RuntimeError(f"Missing required tools: {', '.join(missing)}")
    print(f"[OK] All required tools available: {', '.join(required)}")

In [29]:
def parse_parent_assembly(parent_assembly: str, sample: str, expected_from: int, expected_to: int) -> Tuple[str, Optional[Tuple[int, int]]]:
    """
    Parse parent_assembly, stripping :start-end suffix if present.
    Returns (clean_assembly, (start, end) or None).
    
    Example:
        Input:  'PAN027.chr1.maternal:157835539-159123986'
        Output: ('PAN027.chr1.maternal', (157835539, 159123986))
    """
    validate_identifier(parent_assembly.split(':')[0], "parent_assembly")
    
    match = re.match(r'^(.+):(\d+)-(\d+)$', parent_assembly)
    if match:
        base_asm = match.group(1)
        start = int(match.group(2))
        end = int(match.group(3))
        
        if not base_asm.startswith(f"{sample}."):
            raise RuntimeError(f"parent_assembly '{parent_assembly}' does not start with '{sample}.'")
        
        if start != expected_from or end != expected_to:
            print(f"[WARN] parent_assembly coordinates ({start}-{end}) differ from record ({expected_from}-{expected_to})")
        
        return base_asm, (start, end)
    else:
        # No :start-end suffix
        if not parent_assembly.startswith(f"{sample}."):
            raise RuntimeError(f"parent_assembly '{parent_assembly}' does not start with '{sample}.'")
        return parent_assembly, None


def find_vcf_path(variants_path: str, variants_dir: Path, project_root: Path) -> Path:
    """Find the actual VCF path, handling relative paths and .vcf/.vcf.gz extensions."""
    vcf_path = Path(variants_path)
    
    # Try as absolute or relative to current dir
    if vcf_path.is_absolute():
        candidates = [vcf_path, Path(str(vcf_path) + '.gz')]
    else:
        # Try relative to variants_dir
        candidates = [
            variants_dir / vcf_path,
            variants_dir / (str(vcf_path) + '.gz'),
            project_root / vcf_path,
            project_root / (str(vcf_path) + '.gz'),
        ]
    
    for cand in candidates:
        if cand.exists():
            return cand
    
    # Also try without .gz if path has it
    for cand in list(candidates):
        if str(cand).endswith('.gz'):
            plain = Path(str(cand)[:-3])
            if plain.exists():
                return plain
    
    raise FileNotFoundError(f"VCF file not found for path '{variants_path}'. Tried: {candidates}")


def load_fai_lengths(genomes_dir: Path, assemblies: set) -> Dict[str, int]:
    """Load chromosome lengths from .fai files."""
    lengths = {}
    
    for asm in assemblies:
        # asm is like "PAN027.chr1.maternal" or "PAN027.chr1.paternal"
        parts = asm.split('.')
        if len(parts) >= 3:
            sample = parts[0]
            chrom = parts[1]
            haplo = parts[2]  # maternal or paternal or haplotype1/haplotype2
            
            if haplo == 'maternal':
                fai_path = genomes_dir / sample / 'maternal' / f"{asm}.fasta.fai"
            elif haplo == 'paternal':
                fai_path = genomes_dir / sample / 'paternal' / f"{asm}.fasta.fai"
            elif haplo in ('haplotype1', 'haplotype2'):
                fai_path = genomes_dir / sample / haplo / f"{asm}.fasta.fai"
            else:
                fai_path = genomes_dir / f"{asm}.fasta.fai"
        else:
            fai_path = genomes_dir / f"{asm}.fasta.fai"
        
        if not fai_path.exists():
            raise FileNotFoundError(f".fai file not found for assembly '{asm}': {fai_path}")
        
        with open(fai_path, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    chrom_name = parts[0]
                    chrom_len = int(parts[1])
                    lengths[chrom_name] = chrom_len
    
    print(f"[OK] Loaded lengths for {len(lengths)} assemblies from .fai files")
    return lengths

## 2. Load results.json

Load and validate the block records from `blocks/results.json`.

In [30]:
def load_results_json(results_json: Path) -> List[Dict[str, Any]]:
    """Load and validate blocks/results.json."""
    if not results_json.exists():
        raise FileNotFoundError(f"Results JSON not found: {results_json}")
    
    with open(results_json, 'r') as f:
        data = json.load(f)
    
    if 'data' not in data:
        raise ValueError(f"results.json missing required key 'data'")
    
    records = data['data']
    if not isinstance(records, list):
        raise ValueError(f"results.json 'data' must be a list, got {type(records)}")
    
    required_fields = ['parent_assembly', 'parent_from', 'parent_to', 'variants']
    for i, rec in enumerate(records):
        for field in required_fields:
            if field not in rec:
                raise ValueError(f"Block record {i} missing required field '{field}'")
    
    print(f"[OK] Loaded {len(records)} block records from {results_json}")
    return records


# Load the results
print(f"Loading results from: {RESULTS_JSON}")
records = load_results_json(RESULTS_JSON)
print(f"Total block records: {len(records)}")

Loading results from: /home/meta/tomsko/mutation_rates/blocks/results.json
[OK] Loaded 1891 block records from /home/meta/tomsko/mutation_rates/blocks/results.json
Total block records: 1891


In [31]:
# Show first few records
print("First 3 records:")
for i, rec in enumerate(records[:3]):
    print(f"\nRecord {i}:")
    for k, v in rec.items():
        print(f"  {k}: {v}")

First 3 records:

Record 0:
  grandparent_assembly: PAN010.chr1.haplotype1
  grandparent_from: 160357371
  grandparent_to: 160704346
  parent_assembly: PAN027.chr1.maternal
  parent_from: 163904876
  parent_to: 164251848
  daughter_assembly: PAN028.chr1.haplotype2
  daughter_from: 163444170
  daughter_to: 163791144
  id: 6
  extracted_files: {'grandparent': 'extracted_blocks/6_grandparent.fasta', 'parent': 'extracted_blocks/6_parent.fasta', 'daughter': 'extracted_blocks/6_daughter.fasta'}
  variants: variants/6.vcf
  variants_gp_to_p: 3
  variants_gp_to_p_snv: 0
  variants_gp_to_p_indel: 3
  variants_d_to_p: 1
  variants_d_to_p_snv: 0
  variants_d_to_p_indel: 1
  variants_gp_to_p_filtered: 3
  variants_gp_to_p_filtered_snvs: 0
  variants_gp_to_p_filtered_indels: 3
  variants_removed: 0
  variants_removed_snvs: 0
  variants_removed_indels: 0
  variants_ignored: 0
  variants_ignored_snvs: 0
  variants_ignored_indels: 0
  variants_ignored_027: 0
  variants_ignored_snvs_027: 0
  variants_i

## 3. Collect unique assemblies and lengths

Parse `parent_assembly` for each record and collect unique assemblies, then load their lengths from `.fai` files.

In [32]:
# Collect unique assemblies
unique_assemblies = set()
for rec in records:
    parent_asm, _ = parse_parent_assembly(rec['parent_assembly'], SAMPLE, 
                                           rec['parent_from'], rec['parent_to'])
    unique_assemblies.add(parent_asm)

print(f"Unique assemblies ({len(unique_assemblies)}):")
for asm in sorted(unique_assemblies):
    print(f"  - {asm}")

Unique assemblies (46):
  - PAN027.chr1.maternal
  - PAN027.chr1.paternal
  - PAN027.chr10.maternal
  - PAN027.chr10.paternal
  - PAN027.chr11.maternal
  - PAN027.chr11.paternal
  - PAN027.chr12.maternal
  - PAN027.chr12.paternal
  - PAN027.chr13.maternal
  - PAN027.chr13.paternal
  - PAN027.chr14.maternal
  - PAN027.chr14.paternal
  - PAN027.chr15.maternal
  - PAN027.chr15.paternal
  - PAN027.chr16.maternal
  - PAN027.chr16.paternal
  - PAN027.chr17.maternal
  - PAN027.chr17.paternal
  - PAN027.chr18.maternal
  - PAN027.chr18.paternal
  - PAN027.chr19.maternal
  - PAN027.chr19.paternal
  - PAN027.chr2.maternal
  - PAN027.chr2.paternal
  - PAN027.chr20.maternal
  - PAN027.chr20.paternal
  - PAN027.chr21.maternal
  - PAN027.chr21.paternal
  - PAN027.chr22.maternal
  - PAN027.chr22.paternal
  - PAN027.chr3.maternal
  - PAN027.chr3.paternal
  - PAN027.chr4.maternal
  - PAN027.chr4.paternal
  - PAN027.chr5.maternal
  - PAN027.chr5.paternal
  - PAN027.chr6.maternal
  - PAN027.chr6.paternal


In [33]:
# Load chromosome lengths from .fai files
lengths = load_fai_lengths(GENOMES_DIR, unique_assemblies)

print("\nAssembly lengths:")
for asm in sorted(lengths.keys()):
    print(f"  {asm}: {lengths[asm]:,} bp")

[OK] Loaded lengths for 46 assemblies from .fai files

Assembly lengths:
  PAN027.chr1.maternal: 247,024,268 bp
  PAN027.chr1.paternal: 247,769,562 bp
  PAN027.chr10.maternal: 134,377,690 bp
  PAN027.chr10.paternal: 134,510,189 bp
  PAN027.chr11.maternal: 134,801,169 bp
  PAN027.chr11.paternal: 134,716,119 bp
  PAN027.chr12.maternal: 134,857,675 bp
  PAN027.chr12.paternal: 133,492,916 bp
  PAN027.chr13.maternal: 109,439,428 bp
  PAN027.chr13.paternal: 106,744,078 bp
  PAN027.chr14.maternal: 100,407,886 bp
  PAN027.chr14.paternal: 103,031,843 bp
  PAN027.chr15.maternal: 98,030,055 bp
  PAN027.chr15.paternal: 98,817,628 bp
  PAN027.chr16.maternal: 91,592,334 bp
  PAN027.chr16.paternal: 90,505,374 bp
  PAN027.chr17.maternal: 84,214,608 bp
  PAN027.chr17.paternal: 83,721,253 bp
  PAN027.chr18.maternal: 81,158,592 bp
  PAN027.chr18.paternal: 80,492,778 bp
  PAN027.chr19.maternal: 62,748,151 bp
  PAN027.chr19.paternal: 63,523,024 bp
  PAN027.chr2.maternal: 242,507,399 bp
  PAN027.chr2.patern

## 4. Process blocks in parallel

For each block:
1. Find the VCF path (handling `.vcf` or `.vcf.gz`)
2. Read the header (## lines and #CHROM line)
3. Validate contig line matches expected `parent_assembly:start-end`
4. Transform coordinates: `abs_pos = parent_from + block_pos - 1`
5. Rebuild record with new CHROM = clean parent_assembly and new POS
6. Collect all records and header info

In [34]:
def process_block(record: Dict[str, Any], variants_dir: Path, project_root: Path, 
                  sample: str, temp_dir: str) -> Tuple[Union[int, str], List[str], str, Optional[str], List[str]]:
    """
    Process a single block record.
    Returns (block_id, transformed_records, parent_assembly, header_line, info_format_headers).
    """
    block_id = record.get('block_id', record.get('id', 'unknown'))
    parent_assembly_full = record['parent_assembly']
    parent_from = record['parent_from']
    parent_to = record['parent_to']
    variants_path = record['variants']
    
    if VERBOSE:
        print(f"[BLOCK {block_id}] Processing {parent_assembly_full}:{parent_from}-{parent_to}", file=sys.stderr)
    
    # Parse parent_assembly
    parent_assembly, coords = parse_parent_assembly(parent_assembly_full, sample, parent_from, parent_to)
    
    # Find VCF path
    vcf_path = find_vcf_path(variants_path, variants_dir, project_root)
    
    # Open VCF (handle gzipped or plain)
    if str(vcf_path).endswith('.gz'):
        f = gzip.open(vcf_path, 'rt')
    else:
        f = open(vcf_path, 'r')
    
    try:
        header_lines = []
        info_format_headers = []  # Collect ##INFO and ##FORMAT lines
        contig_line = None
        records_out = []
        header_found = False
        
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('##'):
                if line.startswith('##contig='):
                    contig_line = line
                else:
                    header_lines.append(line)
                    # Collect INFO and FORMAT headers for deduplication across blocks
                    if line.startswith('##INFO=') or line.startswith('##FORMAT='):
                        info_format_headers.append(line)
            elif line.startswith('#CHROM'):
                header_found = True
                header_line = line
                break
        
        if not header_found:
            raise RuntimeError(f"Block {block_id}: No header line found in VCF {vcf_path}")
        
        if contig_line is None:
            raise RuntimeError(f"Block {block_id}: No ##contig= line found in VCF {vcf_path}")
        
        # Validate contig line matches expected
        contig_match = re.search(r'<ID=([^,>]+)', contig_line)
        if contig_match:
            contig_id = contig_match.group(1)
            # Expected format: parent_assembly:start-end
            expected_contig = f"{parent_assembly}:{parent_from}-{parent_to}"
            if contig_id != expected_contig:
                print(f"[WARN] Block {block_id}: Contig ID '{contig_id}' differs from expected '{expected_contig}'", file=sys.stderr)
        
        # Process variant records
        for line in f:
            line = line.rstrip('\n')
            if not line or line.startswith('#'):
                continue
            
            parts = line.split('\t')
            if len(parts) < 8:
                raise RuntimeError(f"Block {block_id}: Invalid VCF record: {line[:100]}")
            
            # Transform coordinates
            chrom = parent_assembly
            pos = parent_from + int(parts[1]) - 1
            
            # Validate position is within bounds
            if pos < parent_from or pos > parent_to:
                raise RuntimeError(f"Block {block_id}: Transformed position {pos} out of range [{parent_from}, {parent_to}]")
            
            # Rebuild record with new CHROM and POS
            new_parts = [chrom, str(pos)] + parts[2:]
            records_out.append('\t'.join(new_parts))
        
        if not records_out:
            print(f"[WARN] Block {block_id}: No variant records found in {vcf_path}", file=sys.stderr)
        
        return (block_id, records_out, parent_assembly, header_line if header_found else None, info_format_headers)
    
    finally:
        f.close()

In [35]:
# Create temp directory for processing
temp_dir = tempfile.mkdtemp(prefix='merge_vcf_')
print(f"Temp directory: {temp_dir}")

# Check required tools
check_required_tools()

print(f"\nProcessing {len(records)} blocks with {THREADS} threads...")

Temp directory: /tmp/merge_vcf_qtugl39o
[OK] All required tools available: bcftools, bgzip, tabix, bedtools, awk, sort

Processing 1891 blocks with 8 threads...


In [36]:
# Process blocks in parallel
all_records = []
header_line = None
all_header_lines = {}  # block_id -> #CHROM header line
all_info_format_headers = []  # Collect ##INFO and ##FORMAT from all blocks

if THREADS > 1:
    print(f"Using ProcessPoolExecutor with {THREADS} workers")
    
    with ProcessPoolExecutor(max_workers=THREADS) as executor:
        futures = {}
        for rec in records:
            fut = executor.submit(process_block, rec, VARIANTS_DIR, PROJECT_ROOT, 
                                 SAMPLE, temp_dir)
            futures[fut] = rec
        
        completed = 0
        for fut in as_completed(futures):
            rec = futures[fut]
            try:
                block_id, block_records, parent_asm, hdr, info_fmt = fut.result()
                if hdr:
                    if header_line is None:
                        header_line = hdr
                    all_header_lines[block_id] = hdr
                all_info_format_headers.extend(info_fmt)
                for line in block_records:
                    parts = line.split('\t')
                    chrom = parts[0]
                    pos = int(parts[1])
                    rest = '\t'.join(parts[2:])
                    all_records.append((chrom, pos, rest))
                completed += 1
                if completed % 100 == 0:
                    print(f"  Processed {completed}/{len(records)} blocks...")
            except Exception as e:
                raise RuntimeError(f"Failed to process block {rec.get('block_id', 'unknown')}: {e}")
else:
    print("Processing blocks sequentially")
    
    for i, rec in enumerate(records):
        block_id, block_records, parent_asm, hdr, info_fmt = process_block(
            rec, VARIANTS_DIR, PROJECT_ROOT, SAMPLE, temp_dir)
        if hdr:
            if header_line is None:
                header_line = hdr
            all_header_lines[block_id] = hdr
        all_info_format_headers.extend(info_fmt)
        for line in block_records:
            parts = line.split('\t')
            chrom = parts[0]
            pos = int(parts[1])
            rest = '\t'.join(parts[2:])
            all_records.append((chrom, pos, rest))
        if (i + 1) % 100 == 0:
            print(f"  Processed {i + 1}/{len(records)} blocks...")

print(f"\nTotal variant records collected: {len(all_records):,}")

Using ProcessPoolExecutor with 8 workers


[BLOCK 6] Processing PAN027.chr1.maternal:163904876-164251848[BLOCK 5] Processing PAN027.chr1.maternal:163170412-163829125[BLOCK 3] Processing PAN027.chr1.maternal:161486563-162273010[BLOCK 0] Processing PAN027.chr1.maternal:157835539-159123986[BLOCK 2] Processing PAN027.chr1.maternal:159904614-161394594


[BLOCK 7] Processing PAN027.chr1.maternal:164225496-165095549[BLOCK 4] Processing PAN027.chr1.maternal:162276411-163193857[BLOCK 1] Processing PAN027.chr1.maternal:159250244-159904592





[BLOCK 10] Processing PAN027.chr1.maternal:172620692-174031726
[BLOCK 8] Processing PAN027.chr1.maternal:165024038-166564613
[BLOCK 11] Processing PAN027.chr1.maternal:174012836-174957302
[BLOCK 14] Processing PAN027.chr1.maternal:178737866-178923354[BLOCK 15] Processing PAN027.chr1.maternal:178909925-179427338[BLOCK 12] Processing PAN027.chr1.maternal:174933642-176514964[BLOCK 13] Processing PAN027.chr1.maternal:176492634-178765662[BLOCK 20] Processing PAN027.chr1.maternal:189230228-189890227[BLOC

  Processed 100/1891 blocks...


[BLOCK 139] Processing PAN027.chr10.maternal:116250009-119147116
[BLOCK 143] Processing PAN027.chr10.maternal:123461863-123607206
[BLOCK 142] Processing PAN027.chr10.maternal:123064683-123427037
[BLOCK 144] Processing PAN027.chr10.maternal:123569478-123612217
[BLOCK 145] Processing PAN027.chr10.maternal:123594083-123950182
[WARN] Block 143: No variant records found in /home/meta/tomsko/mutation_rates/variants/143.vcf
[BLOCK 146] Processing PAN027.chr10.maternal:123927572-124196047
[WARN] Block 137: No variant records found in /home/meta/tomsko/mutation_rates/variants/137.vcf
[BLOCK 141] Processing PAN027.chr10.maternal:119709675-123158535[BLOCK 147] Processing PAN027.chr10.maternal:124173160-125220864

[BLOCK 149] Processing PAN027.chr10.maternal:127725217-127803991
[BLOCK 150] Processing PAN027.chr10.maternal:127776995-128987613
[WARN] Block 144: No variant records found in /home/meta/tomsko/mutation_rates/variants/144.vcf
[BLOCK 148] Processing PAN027.chr10.maternal:125199853-1277450

  Processed 200/1891 blocks...


...

  Processed 1700/1891 blocks...


[BLOCK 1739] Processing PAN027.chr6.paternal:125828329-126670846
[BLOCK 1740] Processing PAN027.chr6.paternal:126660807-127567142
[BLOCK 1742] Processing PAN027.chr6.paternal:130728263-130750232
[BLOCK 1743] Processing PAN027.chr6.paternal:130735032-131334624
[BLOCK 1741] Processing PAN027.chr6.paternal:128240313-130743387
[WARN] Block 1742: No variant records found in /home/meta/tomsko/mutation_rates/variants/1742.vcf
[BLOCK 1744] Processing PAN027.chr6.paternal:131326171-132375966
[BLOCK 1745] Processing PAN027.chr6.paternal:132355592-132724274
[BLOCK 1747] Processing PAN027.chr6.paternal:134248865-134414435
[BLOCK 1746] Processing PAN027.chr6.paternal:132701071-134109219
[BLOCK 1750] Processing PAN027.chr7.paternal:601745-1069736
[BLOCK 1751] Processing PAN027.chr7.paternal:6717473-7145337
[BLOCK 1748] Processing PAN027.chr6.paternal:134392850-137496828
[WARN] Block 1747: No variant records found in /home/meta/tomsko/mutation_rates/variants/1747.vcf
[BLOCK 1749] Processing PAN027.ch

  Processed 1800/1891 blocks...


[BLOCK 1716] Processing PAN027.chr5.paternal:48859622-51203219
[BLOCK 1843] Processing PAN027.chr9.paternal:19217369-19589503
[BLOCK 1842] Processing PAN027.chr9.paternal:15552410-19235315
[BLOCK 1844] Processing PAN027.chr9.paternal:19554849-21338448
[BLOCK 1845] Processing PAN027.chr9.paternal:21316757-21484140
[WARN] Block 1716: No variant records found in /home/meta/tomsko/mutation_rates/variants/1716.vcf
[BLOCK 1847] Processing PAN027.chr9.paternal:23795848-24289625
[BLOCK 1846] Processing PAN027.chr9.paternal:21460874-23816450
[BLOCK 1848] Processing PAN027.chr9.paternal:24270914-27853181
[BLOCK 1850] Processing PAN027.chr9.paternal:63635055-65547527
[BLOCK 1852] Processing PAN027.chr9.paternal:67610085-68248371
[BLOCK 1851] Processing PAN027.chr9.paternal:65525041-67633226
[BLOCK 1853] Processing PAN027.chr9.paternal:68224845-68893080
[BLOCK 1854] Processing PAN027.chr9.paternal:68868193-69579517
[BLOCK 1855] Processing PAN027.chr9.paternal:69658673-70116080
[BLOCK 1856] Process


Total variant records collected: 20,259


## 5. Build merged VCF

1. Deduplicate INFO/FORMAT headers by ID
2. Add contig lines with lengths from .fai
3. Sort records by (CHROM, POS)
4. Write to temp file
5. bgzip and tabix
6. Move to final location

In [37]:
# Validate header consistency across blocks
unique_headers = set(all_header_lines.values())
if len(unique_headers) > 1:
    differing_blocks = [bid for bid, hdr in all_header_lines.items() if hdr != header_line]
    print(f"[WARN] Inconsistent #CHROM header lines across {len(differing_blocks)} blocks")
    print(f"Using header from first block")

print(f"Header line: {header_line}")

Header line: #CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	sample


In [38]:
# Collect header lines from first block
first_rec = records[0]
first_vcf = find_vcf_path(first_rec['variants'], VARIANTS_DIR, PROJECT_ROOT)
header_lines = []

print(f"Reading header from first VCF: {first_vcf}")

if str(first_vcf).endswith('.gz'):
    f = gzip.open(first_vcf, 'rt')
else:
    f = open(first_vcf, 'r')

with f:
    for line in f:
        if line.startswith('##'):
            header_lines.append(line.rstrip('\n'))
        elif line.startswith('#CHROM'):
            break

print(f"Collected {len(header_lines)} header lines")

Reading header from first VCF: /home/meta/tomsko/mutation_rates/variants/6.vcf
Collected 11 header lines


In [39]:
def write_merged_vcf(all_records: List[Tuple[str, int, str]], header_lines: List[str], 
                     header_line: str, lengths: Dict[str, int], out_vcf: Path, 
                     temp_dir: str, all_info_format_headers: Optional[List[str]] = None) -> None:
    """Write merged, sorted VCF with proper header."""
    
    # Get unique assemblies from records
    unique_asms = sorted(set(r[0] for r in all_records))
    
    # Build header
    final_header = []
    
    # Keep non-contig ## lines (excluding INFO/FORMAT which will be deduplicated below)
    for line in header_lines:
        if not line.startswith('##contig=') and not line.startswith('##INFO=') and not line.startswith('##FORMAT='):
            final_header.append(line)
    
    # Deduplicate INFO and FORMAT headers from all blocks
    if all_info_format_headers:
        seen_ids = set()
        for line in all_info_format_headers:
            # Extract ID from ##INFO=<ID=xxx,...> or ##FORMAT=<ID=xxx,...>
            match = re.search(r'ID=([^,>]+)', line)
            if match:
                id_val = match.group(1)
                if id_val not in seen_ids:
                    final_header.append(line)
                    seen_ids.add(id_val)
        print(f"Deduplicated INFO/FORMAT headers: {len(seen_ids)} unique IDs")
    
    # Add contig lines with lengths from .fai
    for asm in unique_asms:
        if asm not in lengths:
            raise RuntimeError(f"Length not found for assembly '{asm}' in .fai files")
        final_header.append(f"##contig=<ID={asm},length={lengths[asm]}>")
    
    # Add header line
    final_header.append(header_line)
    
    # Sort records by (CHROM, POS)
    sorted_records = sorted(all_records, key=lambda r: (r[0], r[1]))
    
    # Write to temp file
    temp_vcf = Path(temp_dir) / 'merged.vcf'
    
    print(f"Writing {len(sorted_records):,} records to temporary VCF...")
    with open(temp_vcf, 'w') as f:
        for line in final_header:
            f.write(line + '\n')
        for chrom, pos, rest in sorted_records:
            f.write(f"{chrom}\t{pos}\t{rest}\n")
    
    # bgzip and index
    print("Compressing with bgzip...")
    gz_path = Path(temp_dir) / 'merged.vcf.gz'
    with open(gz_path, 'wb') as f:
        run(['bgzip', '-c', str(temp_vcf)], stdout=f, text=False)
    
    print("Indexing with tabix...")
    tbi_path = Path(temp_dir) / 'merged.vcf.gz.tbi'
    run(['tabix', '-p', 'vcf', str(gz_path)])
    
    # Atomically move to final location
    final_gz = Path(out_vcf)
    final_tbi = Path(str(out_vcf) + '.tbi')
    
    shutil.move(str(gz_path), str(final_gz))
    shutil.move(str(tbi_path), str(final_tbi))
    
    print(f"[OK] Wrote merged VCF: {final_gz} ({len(sorted_records):,} records)")
    print(f"[OK] Wrote VCF index: {final_tbi}")


# Write the merged VCF
write_merged_vcf(all_records, header_lines, header_line, lengths, OUT_VCF, temp_dir, all_info_format_headers)

Deduplicated INFO/FORMAT headers: 4 unique IDs
Writing 20,259 records to temporary VCF...
Compressing with bgzip...
Indexing with tabix...
[OK] Wrote merged VCF: /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz (20,259 records)
[OK] Wrote VCF index: /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz.tbi


[RUN] bgzip -c /tmp/merge_vcf_qtugl39o/merged.vcf
[RUN] tabix -p vcf /tmp/merge_vcf_qtugl39o/merged.vcf.gz


## 6. Build all_blocks.bed

For each record, write BED line: `clean_assembly\tparent_from-1\tparent_to`
- Sort by chrom, start
- Merge overlapping/adjacent intervals
- Write to `all_blocks.bed`

In [40]:
def write_blocks_bed(records: List[Dict[str, Any]], sample: str, out_bed: Path) -> None:
    """Write all_blocks.bed with 0-based coordinates."""
    
    bed_lines = []
    for rec in records:
        parent_assembly_full = rec['parent_assembly']
        parent_from = rec['parent_from']
        parent_to = rec['parent_to']
        
        # Parse to get clean assembly name
        parent_assembly, _ = parse_parent_assembly(parent_assembly_full, sample, parent_from, parent_to)
        
        # BED format: chrom, start (0-based), end (exclusive)
        bed_start = parent_from - 1
        bed_end = parent_to
        bed_lines.append((parent_assembly, bed_start, bed_end))
    
    # Sort by chrom then start
    bed_lines.sort(key=lambda x: (x[0], x[1]))
    
    # Merge overlapping intervals (simple Python implementation)
    merged = []
    current = None
    
    for chrom, start, end in bed_lines:
        if current is None or current[0] != chrom:
            if current:
                merged.append(current)
            current = (chrom, start, end)
        elif start <= current[2]:
            # Overlapping or adjacent, extend
            current = (chrom, current[1], max(current[2], end))
        else:
            merged.append(current)
            current = (chrom, start, end)
    
    if current:
        merged.append(current)
    
    # Write BED file
    with open(out_bed, 'w') as f:
        for chrom, start, end in merged:
            f.write(f"{chrom}\t{start}\t{end}\n")
    
    print(f"[OK] Wrote all_blocks.bed: {out_bed} ({len(merged)} merged intervals from {len(bed_lines)} blocks)")


# Write the blocks BED
write_blocks_bed(records, SAMPLE, OUT_BED)

[OK] Wrote all_blocks.bed: /home/meta/tomsko/mutation_rates/all_blocks.bed (615 merged intervals from 1891 blocks)


## 7. Optional variant-level filtering

**Note:** Step 2 already applied `--ignore-variants-in-regions` to filter out variants in problematic grandparent regions. This section shows how to apply additional filtering if needed.

To filter the merged VCF by an optional BED (e.g., additional problematic regions), use `bcftools view -T ^bedfile` to exclude variants in those regions.

In [41]:
# OPTIONAL: Filter merged VCF by problematic regions
# Uncomment and modify the paths below if additional filtering is needed

# PROBLEMATIC_BED = PROJECT_ROOT / 'ignore_regions' / 'problematic.PAN027.bed'
# FILTERED_VCF = PROJECT_ROOT / 'merged_variants_filtered.vcf.gz'

# if PROBLEMATIC_BED.exists():
#     print(f"Filtering variants in problematic regions: {PROBLEMATIC_BED}")
#     run([
#         'bcftools', 'view',
#         '-T', f'^{PROBLEMATIC_BED}',  # ^ excludes variants in regions
#         '-Oz', '-o', str(FILTERED_VCF),
#         str(OUT_VCF)
#     ])
#     run(['tabix', '-p', 'vcf', str(FILTERED_VCF)])
#     print(f"[OK] Wrote filtered VCF: {FILTERED_VCF}")
# else:
#     print(f"Problematic BED not found: {PROBLEMATIC_BED}")

print("Skipping optional filtering (step 2 already applied ignore BEDs)")

Skipping optional filtering (step 2 already applied ignore BEDs)


## 8. Verification

Verify the output files:
- Count variants in merged VCF
- List contigs
- Show first few records
- Show all_blocks.bed stats

In [42]:
# Count variants in merged VCF
print("=== Merged VCF Statistics ===")
result = run(['bcftools', 'view', '-H', str(OUT_VCF)], capture=True)
variant_count = len(result.stdout.strip().split('\n'))
print(f"Total variants: {variant_count:,}")

# List contigs
print("\n=== Contigs ===")
result = run(['bcftools', 'view', '-H', str(OUT_VCF)], capture=True)
contigs = set()
for line in result.stdout.strip().split('\n'):
    if line:
        parts = line.split('\t')
        if len(parts) >= 1:
            contigs.add(parts[0])

print(f"Number of contigs: {len(contigs)}")
print("Contigs:", ', '.join(sorted(contigs)))

[RUN] bcftools view -H /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz


=== Merged VCF Statistics ===
Total variants: 20,259

=== Contigs ===


[RUN] bcftools view -H /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz


Number of contigs: 46
Contigs: PAN027.chr1.maternal, PAN027.chr1.paternal, PAN027.chr10.maternal, PAN027.chr10.paternal, PAN027.chr11.maternal, PAN027.chr11.paternal, PAN027.chr12.maternal, PAN027.chr12.paternal, PAN027.chr13.maternal, PAN027.chr13.paternal, PAN027.chr14.maternal, PAN027.chr14.paternal, PAN027.chr15.maternal, PAN027.chr15.paternal, PAN027.chr16.maternal, PAN027.chr16.paternal, PAN027.chr17.maternal, PAN027.chr17.paternal, PAN027.chr18.maternal, PAN027.chr18.paternal, PAN027.chr19.maternal, PAN027.chr19.paternal, PAN027.chr2.maternal, PAN027.chr2.paternal, PAN027.chr20.maternal, PAN027.chr20.paternal, PAN027.chr21.maternal, PAN027.chr21.paternal, PAN027.chr22.maternal, PAN027.chr22.paternal, PAN027.chr3.maternal, PAN027.chr3.paternal, PAN027.chr4.maternal, PAN027.chr4.paternal, PAN027.chr5.maternal, PAN027.chr5.paternal, PAN027.chr6.maternal, PAN027.chr6.paternal, PAN027.chr7.maternal, PAN027.chr7.paternal, PAN027.chr8.maternal, PAN027.chr8.paternal, PAN027.chr9.materna

In [43]:
# Show first few records
print("=== First 10 variant records ===")
result = run(['bcftools', 'view', '-H', str(OUT_VCF), '-n', '10'], capture=True)
for line in result.stdout.strip().split('\n')[:10]:
    print(line)

=== First 10 variant records ===



[RUN] bcftools view -H /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz -n 10


In [44]:
# Show all_blocks.bed stats
print("\n=== all_blocks.bed Statistics ===")
with open(OUT_BED, 'r') as f:
    bed_lines = f.readlines()

print(f"Total merged intervals: {len(bed_lines):,}")

# Calculate total covered length
total_length = 0
for line in bed_lines:
    parts = line.strip().split('\t')
    if len(parts) >= 3:
        start = int(parts[1])
        end = int(parts[2])
        total_length += (end - start)

print(f"Total covered length: {total_length:,} bp ({total_length / 1e6:.2f} Mb)")

# Show first few BED lines
print("\nFirst 10 BED intervals:")
for line in bed_lines[:10]:
    print(line.strip())


=== all_blocks.bed Statistics ===
Total merged intervals: 615
Total covered length: 2,651,508,018 bp (2651.51 Mb)

First 10 BED intervals:
PAN027.chr1.maternal	26616530	30183632
PAN027.chr1.maternal	30193232	31543721
PAN027.chr1.maternal	31580756	38617434
PAN027.chr1.maternal	38618338	39238735
PAN027.chr1.maternal	108317254	114061797
PAN027.chr1.maternal	114106900	115686347
PAN027.chr1.maternal	115814657	118113444
PAN027.chr1.maternal	118287234	120589404
PAN027.chr1.maternal	121235445	127393552
PAN027.chr1.maternal	127999080	128319284


In [45]:
# Verify files exist and are valid
print("\n=== File Verification ===")

files_to_check = [
    (OUT_VCF, "Merged VCF (bgzipped)"),
    (Path(str(OUT_VCF) + '.tbi'), "VCF index"),
    (OUT_BED, "All blocks BED"),
]

for fpath, description in files_to_check:
    if fpath.exists():
        size = fpath.stat().st_size
        print(f"✓ {description}: {fpath.name} ({size:,} bytes)")
    else:
        print(f"✗ {description}: {fpath.name} NOT FOUND")

# Verify VCF is valid bgzip
print("\nValidating VCF with bcftools...")
run(['bcftools', 'view', '-h', str(OUT_VCF)], capture=True)
print("VCF validation: OK")


=== File Verification ===
✓ Merged VCF (bgzipped): merged_variants.vcf.gz (261,393 bytes)
✓ VCF index: merged_variants.vcf.gz.tbi (171,723 bytes)
✓ All blocks BED: all_blocks.bed (24,530 bytes)

Validating VCF with bcftools...
VCF validation: OK


[RUN] bcftools view -h /home/meta/tomsko/mutation_rates/merged_variants.vcf.gz


In [46]:
# Clean up temp directory
print(f"\nCleaning up temp directory: {temp_dir}")
shutil.rmtree(temp_dir, ignore_errors=True)
print("Done!")


Cleaning up temp directory: /tmp/merge_vcf_qtugl39o
Done!
